## CANDIDATE RANKING

In [63]:
import joblib

# Load best model (you saved earlier)
best_model = joblib.load("../models/best_model.pkl")

print("✅ Model loaded!")

✅ Model loaded!


In [64]:
import scipy.sparse as sp
import numpy as np
import pandas as pd

X_test = sp.load_npz("../data/X_test.npz")
df_test = pd.read_csv("../data/resume_jd_test_cleaned.csv")

print("✅ Test data loaded!")
print(X_test.shape)

✅ Test data loaded!
(1759, 10001)


In [65]:
import joblib

le = joblib.load("../models/label_encoder.pkl")

print("Classes:", le.classes_)

Classes: ['Good Fit' 'No Fit' 'Potential Fit']


### 🔹 Ranking Factors

We use three main components:

1. **Confidence Score**
   - Obtained from model prediction probabilities
   - Indicates how certain the model is about its prediction

2. **Cosine Similarity**
   - Based on TF-IDF vectors
   - Measures keyword-level similarity between resume and job description

3. **BERT Similarity**
   - Based on semantic embeddings
   - Captures contextual meaning between texts

In [66]:
y_pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)

print("✅ Predictions done!")

✅ Predictions done!


In [67]:
df_ranking = df_test.copy()

# Convert numeric labels → original labels
df_ranking['predicted_label'] = le.inverse_transform(y_pred)

# Confidence score (highest probability)
df_ranking['confidence'] = proba.max(axis=1)

df_ranking.head()

,resume_text,job_description_text,label,resume_clean,jd_clean,cosine_similarity,bert_similarity,predicted_label,confidence
0,Summary7+ years of experience as a BI develope...,Key Responsibilities:Create intricate wiring n...,No Fit,summary7 year experience developer proven trac...,key responsibility create intricate wiring net...,0.035738,0.564143,No Fit,0.963879
1,Professional BackgroundAnalyst versed in data ...,Personal development and becoming the best you...,No Fit,professional backgroundanalyst versed data ana...,personal development becoming best growth expl...,0.100394,0.631430,No Fit,0.894099
2,Executive ProfileDedicated professional with t...,"Location: Tampa, FL\r\nExp: 7-10 Yrs\r\nSPOC: ...",No Fit,executive profilededicated professional accomp...,location tampa exp yr spoc tushar kshirsagar k...,0.045090,0.631957,No Fit,0.978085
3,"Summarytyee\r\nHighlightsMicrosoft Excel, Word...","Primary Location: Melbourne, Florida\r\nV-Soft...",No Fit,summarytyee highlightsmicrosoft excel word out...,primary location melbourne florida soft consul...,0.050667,0.465062,No Fit,0.616572
4,SummaryEIT certified Engineer and ASTQB Certif...,At Oregon Specialty Group the Accounting & Pay...,No Fit,summaryeit certified engineer astqb certified ...,oregon specialty group accounting payroll depa...,0.082104,0.645818,No Fit,0.812609


## sample example

In [68]:
df_ranking.loc[18, [
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity'
]]

predicted_label      Potential Fit
confidence                0.482337
cosine_similarity         0.195399
bert_similarity           0.784537
Name: 18, dtype: object

In [69]:
df_ranking['final_score'] = (
    0.5 * df_ranking['confidence'] +
    0.3 * df_ranking['bert_similarity'] +
    0.2 * df_ranking['cosine_similarity']
)

In [70]:
def assign_tier(label):
    if label == 'Good Fit':
        return 1
    elif label == 'Potential Fit':
        return 2
    else:
        return 3

df_ranking['tier'] = df_ranking['predicted_label'].apply(assign_tier)

In [71]:
df_ranking = df_ranking.sort_values(
    ['tier', 'final_score'],
    ascending=[True, False]
).reset_index(drop=True)

df_ranking['rank'] = df_ranking.index + 1

In [72]:
df_ranking[[
    'rank',
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity',
    'final_score'
]].head(10)

,rank,predicted_label,confidence,cosine_similarity,bert_similarity,final_score
0,1,Good Fit,0.924685,0.111224,0.746746,0.708611
1,2,Good Fit,0.910140,0.133801,0.709332,0.694630
2,3,Good Fit,0.895033,0.124974,0.726506,0.690463
3,4,Good Fit,0.907123,0.124725,0.703598,0.689586
4,5,Good Fit,0.864215,0.161447,0.739281,0.686181
5,6,Good Fit,0.931375,0.124671,0.642416,0.683346
6,7,Good Fit,0.885732,0.066414,0.750909,0.681421
7,8,Good Fit,0.836072,0.113114,0.789218,0.677424
8,9,Good Fit,0.860123,0.093390,0.746740,0.672762
9,10,Good Fit,0.886114,0.103685,0.688705,0.670406


### 🔹 Final Scoring Formula

The final ranking score is calculated as:

#### Explanation:
- **0.5 (Confidence):** Most important, reflects model certainty  
- **0.3 (BERT):** Captures semantic meaning (very important)  
- **0.2 (Cosine):** Captures keyword overlap (less important)

---

### 🔹 Tier-Based Ranking Logic

Candidates are categorized into tiers based on predicted labels:

| Label          | Tier | Meaning              |
|---------------|------|----------------------|
| Good Fit      | 1    | Highly suitable      |
| Potential Fit | 2    | Moderately suitable  |
| No Fit        | 3    | Not suitable         |



# TOP 10

In [73]:
df_ranking[[
    'rank',
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity',
    'final_score'
]].head(10)

,rank,predicted_label,confidence,cosine_similarity,bert_similarity,final_score
0,1,Good Fit,0.924685,0.111224,0.746746,0.708611
1,2,Good Fit,0.910140,0.133801,0.709332,0.694630
2,3,Good Fit,0.895033,0.124974,0.726506,0.690463
3,4,Good Fit,0.907123,0.124725,0.703598,0.689586
4,5,Good Fit,0.864215,0.161447,0.739281,0.686181
5,6,Good Fit,0.931375,0.124671,0.642416,0.683346
6,7,Good Fit,0.885732,0.066414,0.750909,0.681421
7,8,Good Fit,0.836072,0.113114,0.789218,0.677424
8,9,Good Fit,0.860123,0.093390,0.746740,0.672762
9,10,Good Fit,0.886114,0.103685,0.688705,0.670406


In [74]:
# BOTTOM 10

In [75]:
df_ranking[[
    'rank',
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity',
    'final_score'
]].tail(10)

,rank,predicted_label,confidence,cosine_similarity,bert_similarity,final_score
1749,1750,No Fit,0.464541,0.015310,0.371579,0.346806
1750,1751,No Fit,0.392655,0.097204,0.429997,0.344767
1751,1752,No Fit,0.467192,0.038116,0.342089,0.343846
1752,1753,No Fit,0.402399,0.050285,0.435656,0.341953
1753,1754,No Fit,0.474276,0.004055,0.304079,0.329173
1754,1755,No Fit,0.455601,0.058255,0.296058,0.328269
1755,1756,No Fit,0.375419,0.100033,0.396074,0.326538
1756,1757,No Fit,0.451485,0.023277,0.289715,0.317313
1757,1758,No Fit,0.500947,0.027869,0.194295,0.314336
1758,1759,No Fit,0.488919,0.011588,0.205259,0.308355


### 🔹 Conclusion

This ranking approach ensures:
- Better prioritization of candidates
- Combination of model prediction and semantic understanding
- Improved accuracy in resume-job matching